In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor

# =========================
# CONFIG
# =========================
FILE = Path("Ventas 2026.xlsx")
OUT_DIR = Path("outputs_forecast_12w_percentiles")
OUT_DIR.mkdir(exist_ok=True)

YEARS = [2024, 2025]
TOP_K = 5
FORWARD_WEEKS = 12

# Features temporales + memoria
LAGS = [1, 2, 4]     # <- agregado lag_4 (mejora estabilidad)
ROLLS = [4, 8]       # <- opcional, pero suele ayudar

MODEL_PARAMS = dict(
    n_estimators=500,
    random_state=42,
    min_samples_leaf=2,
    n_jobs=-1
)

# Percentiles para escenarios
P_LOW, P_MID, P_HIGH = 10, 50, 90

# =========================
# Helpers
# =========================
def build_features_row(week_start, hist_qty, first_week):
    """Features para una semana (histórica o futura) según historial qty."""
    weekofyear = int(week_start.isocalendar().week)
    month = int(week_start.month)
    week_idx = int((week_start - first_week).days // 7)

    feats = {
        "week_idx": week_idx,
        "weekofyear": weekofyear,
        "month": month
    }

    for lag in LAGS:
        feats[f"lag_{lag}"] = float(hist_qty[-lag]) if len(hist_qty) >= lag else np.nan

    for r in ROLLS:
        if len(hist_qty) >= r:
            feats[f"roll_{r}"] = float(np.mean(hist_qty[-r:]))
        else:
            feats[f"roll_{r}"] = float(np.mean(hist_qty)) if len(hist_qty) > 0 else np.nan

    return feats

def make_future_weeks(last_week, n=12):
    return [last_week + pd.Timedelta(days=7*(k+1)) for k in range(n)]

def rf_tree_percentiles(model, X_row, p_low=10, p_mid=50, p_high=90):
    """Devuelve percentiles usando predicción de cada árbol del RF."""
    # X_row: DataFrame con 1 fila
    preds = np.array([est.predict(X_row)[0] for est in model.estimators_], dtype=float)
    lo = np.percentile(preds, p_low)
    mid = np.percentile(preds, p_mid)
    hi = np.percentile(preds, p_high)
    return float(lo), float(mid), float(hi)

# =========================
# 1) Load + clean
# =========================
df = pd.read_excel(FILE)

required_cols = {"fecha", "Categoría", "cantidad"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas: {missing}. Encontradas: {list(df.columns)}")

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.dropna(subset=["fecha"]).copy()
df = df[df["fecha"].dt.year.isin(YEARS)].copy()

df["Categoría"] = (df["Categoría"].astype(str)
                   .str.replace("\u00A0", " ", regex=False)
                   .str.strip()
                   .str.title())

df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0)

# Semana operativa lun-sáb
df["week_start"] = df["fecha"] - pd.to_timedelta(df["fecha"].dt.weekday, unit="D")
df["dow"] = df["fecha"].dt.weekday
df = df[df["dow"] <= 5].copy()

# Semanas completas del local (>=6 días con ventas)
oper = (df.groupby("week_start")["fecha"].nunique().reset_index(name="days_open"))
full_weeks = set(oper.loc[oper["days_open"] >= 6, "week_start"])
df = df[df["week_start"].isin(full_weeks)].copy()

# =========================
# 2) Weekly aggregation por categoría
# =========================
weekly = (
    df.groupby(["Categoría", "week_start"], as_index=False)
      .agg(qty=("cantidad", "sum"))
      .sort_values(["Categoría", "week_start"])
      .reset_index(drop=True)
)

# Completar semanas faltantes por categoría (continuidad)
all_weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq="W-MON")
cats = weekly["Categoría"].unique()

full = (
    pd.MultiIndex.from_product([cats, all_weeks], names=["Categoría", "week_start"])
      .to_frame(index=False)
)

weekly = full.merge(weekly, on=["Categoría", "week_start"], how="left")
weekly["qty"] = pd.to_numeric(weekly["qty"], errors="coerce").fillna(0)
weekly = weekly.sort_values(["Categoría", "week_start"]).reset_index(drop=True)

# Mantener solo semanas completas del local
weekly = weekly[weekly["week_start"].isin(full_weeks)].copy()

# =========================
# 3) Elegir TOP_K (y forzar Salado si existe)
# =========================
totals = weekly.groupby("Categoría")["qty"].sum().sort_values(ascending=False)
top_cats = list(totals.head(TOP_K).index)

if "Salado" in weekly["Categoría"].unique() and "Salado" not in top_cats:
    top_cats.append("Salado")

print("Categorías a forecastear:", top_cats)

# =========================
# 4) Forecast + escenarios (P10/P50/P90)
# =========================
out_rows = []

# para entrenar sin ruido inicial
start_i = max(max(LAGS), max(ROLLS))

for cat in top_cats:
    w = weekly[weekly["Categoría"] == cat].copy().sort_values("week_start").reset_index(drop=True)

    weeks_hist = list(w["week_start"])
    qty_hist = list(w["qty"].astype(float))

    first_week = weeks_hist[0]
    last_week = weeks_hist[-1]

    # --- Entrenamiento con TODO el histórico pero desde start_i ---
    X_train, y_train = [], []
    for i in range(start_i, len(weeks_hist)):
        feats = build_features_row(weeks_hist[i], qty_hist[:i], first_week)
        X_train.append(feats)
        y_train.append(qty_hist[i])

    X_train = pd.DataFrame(X_train).dropna()
    y_train = np.asarray(y_train[:len(X_train)], dtype=float)

    model = RandomForestRegressor(**MODEL_PARAMS)
    model.fit(X_train, y_train)

    # --- Forecast iterativo con percentiles por semana ---
    future_weeks = make_future_weeks(last_week, FORWARD_WEEKS)

    preds_low, preds_mid, preds_high = [], [], []
    temp_qty = qty_hist.copy()

    for fw in future_weeks:
        feats = build_features_row(fw, temp_qty, first_week)
        X_row = pd.DataFrame([feats]).dropna(axis=1, how="all")

        # Asegurar mismas columnas que en training (por si dropna afectó algo)
        X_row = X_row.reindex(columns=X_train.columns)

        lo, mid, hi = rf_tree_percentiles(model, X_row, P_LOW, P_MID, P_HIGH)

        lo = max(0.0, lo)
        mid = max(0.0, mid)
        hi = max(0.0, hi)

        preds_low.append(lo)
        preds_mid.append(mid)
        preds_high.append(hi)

        # alimentar siguiente paso con la mediana (P50)
        temp_qty.append(mid)

    # Guardar a tabla
    for fw, lo, mid, hi in zip(future_weeks, preds_low, preds_mid, preds_high):
        out_rows.append({
            "Categoría": cat,
            "week_start": fw,
            "forecast_p10": lo,
            "forecast_p50": mid,
            "forecast_p90": hi
        })

    # --- Plot (últimas 40 semanas + banda) ---
    tail_n = 40
    hist_plot_weeks = weeks_hist[-tail_n:]
    hist_plot_qty = qty_hist[-tail_n:]

    plt.figure(figsize=(11, 5.5))
    plt.plot(hist_plot_weeks, hist_plot_qty, label="Histórico (qty)")
    plt.plot(future_weeks, preds_mid, label=f"Forecast P{P_MID}")
    plt.fill_between(future_weeks, preds_low, preds_high, alpha=0.2, label=f"Rango P{P_LOW}-P{P_HIGH}")

    plt.title(f"Forecast {FORWARD_WEEKS} semanas (percentiles) - {cat} (semanal lun-sáb)")
    plt.xlabel("Semana (lunes)")
    plt.ylabel("Cantidad")
    plt.tight_layout()

    img_path = OUT_DIR / f"forecast_{FORWARD_WEEKS}w_percentiles_{cat}.png"
    plt.savefig(img_path, dpi=220)
    plt.close()
    print("✅ Imagen creada:", img_path)

# =========================
# 5) Export
# =========================
out_df = pd.DataFrame(out_rows).sort_values(["Categoría", "week_start"])
csv_path = OUT_DIR / "forecast_12w_percentiles.csv"
out_df.to_csv(csv_path, index=False)

print("\n📌 CSV guardado en:", csv_path.resolve())
print("✅ Listo. Archivos en:", OUT_DIR.resolve())